# Enterprise Track 5 · Monitoring and the Platform's Own Records

**Goal:** watch the data and the pipeline without writing every check yourself: data profiling and anomaly detection on the tables, system tables for lineage and run history, and a path into Prometheus, Grafana and PagerDuty.

> ⚠️ **Mock data only.** This track runs on the agency workspace, which is approved for criminal justice information, but every table here holds the course's fictional records. Real tables come later, with the data owner's approval (see `enterprise/README.md`).

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and checks your settings.
STEP = 5

import importlib
import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import config, enterprise as ent
importlib.reload(config)                                         # picks up edits to lnl/config.py without a restart
N = ent.prepare(STEP, "Monitoring")

## 1. Two kinds of monitoring
| | Checks you write | Checks the platform runs |
|---|---|---|
| Examples | expectations, `validate_quality` | data profiling, anomaly detection |
| Catches | what you predicted could go wrong | drift and gaps you didn't predict |
| Fails a job? | yes, by design | no: you alert on the results |

You need both. The course's checks encode business rules (a crime against a person carries a violent code). The platform's checks notice that yesterday's delivery was a third the usual size, or that the share of blank custody levels doubled.

## 2. Data profiling on silver
A **data profile** computes summary statistics per column (nulls, distinct values, distributions) on a schedule or on demand, and keeps the history in two tables in your schema: profile metrics and drift metrics. It runs on serverless compute, not on this notebook's compute.

One governance catch: the profile tables are new tables, and they don't inherit the masks on the table they describe. Profiling a race column writes its values and counts into the metrics table. That's fine on mock data; on real data, keep the profile's output schema as restricted as the source.

In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
schema_info = w.schemas.get(full_name=N["fq"])
table_info = w.tables.get(full_name=f"{N['fq']}.silver_offenders")
assets_dir = f"/Workspace/Users/{ent.user_email()}/lnl_quality/silver_offenders"

if hasattr(w, "data_quality"):
    from databricks.sdk.service.dataquality import DataProfilingConfig, Monitor, Refresh, SnapshotConfig
    try:
        monitor = w.data_quality.get_monitor(object_type="table", object_id=table_info.table_id)
        print("Profile already exists")
    except Exception:
        monitor = w.data_quality.create_monitor(monitor=Monitor(
            object_type="table", object_id=table_info.table_id,
            data_profiling_config=DataProfilingConfig(output_schema_id=schema_info.schema_id,
                                                      assets_dir=assets_dir, snapshot=SnapshotConfig())))
        print("Profile created")
    try:
        refresh = w.data_quality.create_refresh(object_type="table", object_id=table_info.table_id,
                                                refresh=Refresh(object_type="table", object_id=table_info.table_id))
        print("Refresh started:", refresh.refresh_id)
    except Exception as exc:   # a new profile may already be running its first refresh
        print("Refresh not started:", str(exc).splitlines()[0][:160])
    print("Profile metrics table:", monitor.data_profiling_config.profile_metrics_table_name)
else:   # older SDKs: the previous API for the same feature
    from databricks.sdk.service.catalog import MonitorSnapshot
    try:
        w.quality_monitors.create(table_name=f"{N['fq']}.silver_offenders", assets_dir=assets_dir,
                                  output_schema_name=N["fq"], snapshot=MonitorSnapshot())
    except Exception as exc:
        print("Create:", str(exc).splitlines()[0][:160])
    print("Refresh started:", w.quality_monitors.run_refresh(table_name=f"{N['fq']}.silver_offenders").refresh_id)

The first refresh takes several minutes. When it finishes, the table's **Quality** tab in Catalog Explorer shows the generated dashboard, and the metrics are rows you can query: run the next cell then.

In [ ]:
metrics = f"{N['fq']}.silver_offenders_profile_metrics"
try:
    display(ent.sql(f"""
        SELECT column_name, num_nulls, percent_null, distinct_count, window
        FROM {metrics}
        WHERE column_name IN ('unit_name', 'race_code', 'custody_level', 'restitution_owed', 'stg_status')
        ORDER BY window.start DESC, column_name LIMIT 20"""))
except Exception as exc:
    print(f"Not ready yet ({type(exc).__name__}). Run this cell again when the refresh finishes.")

## 3. Anomaly detection on the whole schema
**Anomaly detection** (Public Preview) watches every table in a schema for **freshness** (did it update when it usually does?) and **completeness** (did about as many rows arrive as usual?), learning each table's normal pattern. It needs `MANAGE` on the schema, which you have as its owner. Turn it on in **Catalog** → your schema → **Details** → **Data Quality Monitoring** → **Enable**. Results land in a system table:

In [ ]:
try:
    display(ent.sql(f"""
        SELECT * FROM system.data_quality_monitoring.table_results
        WHERE catalog_name = '{N['catalog']}' AND schema_name = '{N['schema']}'
        ORDER BY event_time DESC LIMIT 20"""))
except Exception as exc:
    print("Not available:", str(exc).splitlines()[0][:200])

## 4. System tables: the platform's own records
System tables are read-only tables the platform maintains about itself. Three are worth knowing first:

In [ ]:
# Lineage: what reads from what, recorded automatically
try:
    display(ent.sql(f"""
        SELECT source_table_full_name, target_table_full_name, entity_type, max(event_time) AS last_seen
        FROM system.access.table_lineage
        WHERE target_table_full_name LIKE '{N['fq']}.%' OR source_table_full_name LIKE '{N['fq']}.%'
        GROUP BY ALL ORDER BY target_table_full_name"""))
except Exception as exc:
    print("No access to system.access.table_lineage:", str(exc).splitlines()[0][:160])

In [ ]:
# Run history for your job: the same data the Prometheus exporter in section 6 reads.
# System tables lag by up to an hour, so this morning's runs from notebook 04 may not be here yet.
job_ids = [j.job_id for j in w.jobs.list(name=N["job"])]
if job_ids:
    try:
        display(ent.sql(f"""
            SELECT run_id, max(result_state) AS result_state, max(termination_code) AS termination_code,
                   min(period_start_time) AS started, max(period_end_time) AS ended,
                   timestampdiff(SECOND, min(period_start_time), max(period_end_time)) AS seconds
            FROM system.lakeflow.job_run_timeline       -- long runs span several hourly rows, so group by run
            WHERE job_id = '{job_ids[0]}'
            GROUP BY run_id ORDER BY started DESC LIMIT 20"""))
    except Exception as exc:
        print("No access to system.lakeflow:", str(exc).splitlines()[0][:160])

In [ ]:
# Audit: who read the published view (admins usually hold this; skip if you don't have access)
try:
    display(ent.sql(f"""
        SELECT event_time, user_identity.email AS who, action_name,
               request_params.full_name_arg AS object
        FROM system.access.audit
        WHERE event_date >= current_date() - INTERVAL 7 DAYS
          AND request_params.full_name_arg LIKE '{N['fq']}.%'
        ORDER BY event_time DESC LIMIT 20"""))
except Exception as exc:
    print("No access to system.access.audit:", str(exc).splitlines()[0][:160])

## 5. Your own quality history
Every run of `validate_quality` appended its results, so `quality_results` is already a time series. This is the table a dashboard or Grafana panel reads.

In [ ]:
display(ent.sql(f"""
    SELECT date_trunc('HOUR', checked_at) AS checked, run_id,
           count_if(status = 'PASS') AS passed, count_if(status = 'FAIL') AS failed
    FROM {N['fq']}.quality_results
    GROUP BY ALL ORDER BY checked DESC"""))

## 6. Into Prometheus, Grafana and PagerDuty
| Path | What it gives you | Cost and caveats |
|---|---|---|
| **Job notifications → PagerDuty** (notebook 04) | an incident the moment a check fails | none; a workspace admin creates the destination once |
| **[grafana/databricks-prometheus-exporter](https://github.com/grafana/databricks-prometheus-exporter)** → Prometheus → Grafana, Alertmanager → PagerDuty | job, pipeline, SQL warehouse and billing metrics from the system tables, next to your other Prometheus metrics | open source (Apache 2.0) but a young project; runs as a container with a service principal and a SQL warehouse; evaluate before relying on it |
| **Grafana's Databricks data source** | Grafana panels that query `quality_results` and system tables directly through a SQL warehouse | an Enterprise plugin: needs Grafana Enterprise or Grafana Cloud Pro |
| **Databricks SQL alerts and dashboards** | alerts on any query (for example, failing checks today > 0) to the same notification destinations | built in; nothing to install |

A sensible order: job notifications to PagerDuty now (immediate, free), a Databricks SQL dashboard over `quality_results` for the data owner, then the exporter into your existing Prometheus and Grafana once you've tested it with a service principal.

## 7. Clean up
Everything lives in your schema, plus the pipeline, the job and the profile, which are named after you. Set `CONFIRM` to `True` to delete it all.

In [ ]:
CONFIRM = False
if CONFIRM:
    try:   # the profile first, while the table it watches still exists
        if hasattr(w, "data_quality"):
            w.data_quality.delete_monitor(object_type="table", object_id=table_info.table_id)
        else:
            w.quality_monitors.delete(table_name=f"{N['fq']}.silver_offenders")
    except Exception as exc:
        print("Profile:", str(exc).splitlines()[0][:160])
    for job in w.jobs.list(name=N["job"]):
        w.jobs.delete(job.job_id)
    for p in w.pipelines.list_pipelines(filter=f"name LIKE '{N['pipeline']}'"):
        w.pipelines.delete(p.pipeline_id)
    ent.sql(f"DROP SCHEMA IF EXISTS {N['fq']} CASCADE")
    print("Deleted the job, the pipeline, the profile and the schema (tables, views, volume and functions).")
else:
    print("Nothing deleted. Set CONFIRM = True to remove everything this track created.")